In [1]:
import os
import json
import re

from dotenv import load_dotenv
import gradio as gr

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\Parag Choudhary\projects\saffron-grand-indore\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Parag Choudhary\AppData\Local\Temp\ipykernel_103632\1203555022.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "Missing OPENAI_API_KEY. Add it to a .env file in this folder "
        "as OPENAI_API_KEY=your_key_here, then restart the kernel."
    )

model = "gpt-5-nano"
KNOWLEDGE_BASE_DIR = "knowledge_base"
DB_NAME = "vector_db"

if not os.path.isdir(KNOWLEDGE_BASE_DIR):
    raise RuntimeError(
        f"Couldn't find the '{KNOWLEDGE_BASE_DIR}' folder. It should sit "
        "next to this notebook and contain your .md files plus "
        "roohani_threads_structured_data.json."
    )

In [3]:
prose_loader=DirectoryLoader(
    KNOWLEDGE_BASE_DIR,
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
)
prose_docs=prose_loader.load()

for doc in prose_docs:
    filename=os.path.basename(doc.metadata["source"])
    doc.metadata["doctype"]=os.path.splitext(filename)[0]

splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=150)
prose_chunks=splitter.split_documents(prose_docs)


print(f"Loaded {len(prose_docs)} prose file(s) -> split into {len(prose_chunks)} chunks")



Loaded 1 prose file(s) -> split into 2 chunks


In [4]:
def json_to_markdown(data, source_name="saffron_grand_structured_data.json"):
    docs = []

    # Rooms
    for room in data["rooms"]:
        content = (
            f"Room Type: {room['room_type']}\n"
            f"Price per night: ₹{room['price_inr_per_night']}\n"
            f"Capacity: {room['capacity']} guests\n"
            f"Amenities: {', '.join(room['amenities'])}"
        )
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "doc_type": "room",
                    "name": room["room_type"],
                    "source": source_name,
                },
            )
        )

    # Restaurants
    for restaurant in data["restaurants"]:
        content = (
            f"Restaurant: {restaurant['name']}\n"
            f"Cuisine: {restaurant['cuisine']}\n"
            f"Role: {restaurant['role']}\n"
            f"Hours: {restaurant['hours']}\n"
            f"Online Ordering: {restaurant['orderable_online']}"
        )
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "doc_type": "restaurant",
                    "name": restaurant["name"],
                    "source": source_name,
                },
            )
        )

    # Menu Items
    for item in data["menu"]:
        content = (
            f"Item: {item['item']}\n"
            f"Category: {item['category']}\n"
            f"Price: ₹{item['price_inr']}\n"
            f"Vegetarian: {item['veg']}\n"
            f"Spice Level: {item['spice_level']}\n"
            f"Description: {item['description']}"
        )
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "doc_type": "menu_item",
                    "name": item["item"],
                    "category": item["category"],
                    "source": source_name,
                },
            )
        )

    # Room Service
    room_service = data["room_service"]
    content = (
        f"Room Service Available: {room_service['available']}\n"
        f"Hours: {room_service['hours']}\n"
        f"Menu Source: {room_service['menu_source']}\n"
        f"Delivery Charge: ₹{room_service['delivery_charge_inr']}\n"
        f"Average Delivery Time: {room_service['average_delivery_time_minutes']} minutes"
    )
    docs.append(
        Document(
            page_content=content,
            metadata={
                "doc_type": "room_service",
                "source": source_name,
            },
        )
    )

    # Spa Treatments
    for treatment in data["spa"]:
        content = (
            f"Treatment: {treatment['treatment']}\n"
            f"Duration: {treatment['duration_minutes']} minutes\n"
            f"Price: ₹{treatment['price_inr']}"
        )
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "doc_type": "spa_treatment",
                    "name": treatment["treatment"],
                    "source": source_name,
                },
            )
        )

    # Facilities
    facilities = data["facilities"]
    content = (
        f"Pool: {facilities['pool']}\n"
        f"Gym: {facilities['gym']}\n"
        f"Banquet Halls: {facilities['banquet_halls']}\n"
        f"Business Center: {facilities['business_center']}\n"
        f"Airport Transfer: {facilities['airport_transfer']}"
    )
    docs.append(
        Document(
            page_content=content,
            metadata={
                "doc_type": "facilities",
                "source": source_name,
            },
        )
    )

    # Policies
    policies = data["policies"]
    content = (
        f"Check-in Time: {policies['check_in_time']}\n"
        f"Check-out Time: {policies['check_out_time']}\n"
        f"Cancellation Policy: {policies['cancellation_policy']}\n"
        f"Pet Policy: {policies['pet_policy']}\n"
        f"Extra Bed Charge: ₹{policies['extra_bed_charge_inr']}\n"
        f"ID Proof Required: {policies['id_proof_required']}\n"
        f"Minimum Check-in Age: {policies['minimum_check_in_age']}"
    )
    docs.append(
        Document(
            page_content=content,
            metadata={
                "doc_type": "policy",
                "source": source_name,
            },
        )
    )

    # Staff
    for staff in data["staff"]:
        content = (
            f"Name: {staff['name']}\n"
            f"Role: {staff['role']}\n"
            f"Experience: {staff['experience_years']} years"
        )
        docs.append(
            Document(
                page_content=content,
                metadata={
                    "doc_type": "staff",
                    "name": staff["name"],
                    "source": source_name,
                },
            )
        )

    # Location
    location = data["location"]
    content = (
        f"Address: {location['address']}\n"
        f"Distance from Railway Station: {location['distance_from_railway_station_km']} km\n"
        f"Distance from Airport: {location['distance_from_airport_km']} km\n"
        f"Nearby Landmarks: {location['nearby_landmarks']}"
    )
    docs.append(
        Document(
            page_content=content,
            metadata={
                "doc_type": "location",
                "source": source_name,
            },
        )
    )

    # Parking
    docs.append(
        Document(
            page_content=f"Parking Information: {data['parking']}",
            metadata={
                "doc_type": "parking",
                "source": source_name,
            },
        )
    )

    # Contact
    contact = data["contact"]
    content = (
        f"Phone: {contact['phone']}\n"
        f"Email: {contact['email']}\n"
        f"Website: {contact['website']}"
    )
    docs.append(
        Document(
            page_content=content,
            metadata={
                "doc_type": "contact",
                "source": source_name,
            },
        )
    )

    return docs

In [5]:
json_path=os.path.join(KNOWLEDGE_BASE_DIR,"saffron_grand_structured_data.json")

try:
    with open(json_path, "r", encoding="utf-8") as f:
        catalog_data = json.load(f)
except FileNotFoundError:
    raise RuntimeError(
        f"Couldn't find '{json_path}'. Check that roohani_threads_structured_data.json "
        "is inside the knowledge_base folder."
    )
except json.JSONDecodeError as e:
    raise RuntimeError(
        f"'{json_path}' isn't valid JSON ({e}). Check for a missing comma, "
        "quote, or bracket."
    )
catalog_docs = json_to_markdown(catalog_data)
print(f"Loaded {len(catalog_docs)} structured documents")


Loaded 33 structured documents


In [6]:
all_chunks = catalog_docs + prose_chunks

EMBED_MODEL = "multi-qa-MiniLM-L6-cos-v1"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Vectorstore built with {vectorstore._collection.count()} documents")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2756.65it/s]


Vectorstore built with 35 documents


In [7]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

load_dotenv()

all_chunks = catalog_docs + prose_chunks

EMBED_MODEL = "text-embedding-3-small"

embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL
)

if os.path.exists(DB_NAME):
    Chroma(
        persist_directory=DB_NAME,
        embedding_function=embeddings
    ).delete_collection()

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Vectorstore built with {vectorstore._collection.count()} documents")

Vectorstore built with 35 documents


In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-nano"
)

In [9]:
SYSTEM_PROMPT_TEMPLATE = """You are the virtual assistant for The Saffron Grand, a luxury 5-star hotel located in Vijay Nagar, Indore. You help guests and prospective guests with questions about rooms, dining, room service, spa treatments, hotel facilities, events, policies, location, parking, and contact information.

TONE
Be warm, professional, and welcoming — like a knowledgeable hotel concierge. Keep answers concise, helpful, and conversational. Make guests feel informed and taken care of without sounding overly formal or robotic.

GROUNDING RULES (strict)

* Answer using ONLY the information in the "Context" section below. Do not use outside knowledge about hotels, tourism, restaurants, travel, local attractions, or hospitality services unless explicitly stated in the context.
* Never invent or guess room prices, menu prices, amenities, operating hours, spa services, policies, contact details, or any other hotel information that is not present in the context.
* If the context only partially answers a question, answer the part you can and clearly state what information is unavailable.
* If the context contains no relevant information, say so plainly and invite the guest to ask about rooms, dining, spa services, facilities, policies, location, parking, or contact details instead.
* Do not mention that you are using "context", "retrieved documents", "knowledge base", or any internal system.
* If a guest asks about availability, reservations, room inventory, current occupancy, event bookings, restaurant table availability, or real-time status, explain that you do not have access to live booking information.
* If a question refers to a specific room, restaurant, facility, treatment, or menu item, provide only the details available in the context.
* If multiple room categories, restaurants, spa treatments, or menu items match the guest's request, list the relevant options clearly instead of assuming which one they mean.
* If the context includes verified calculations or computed information, state the result directly without recomputing it.

CONVERSATION

* You may be shown earlier turns of the conversation. Use them to resolve references such as "that room", "the suite", "that restaurant", or "the same treatment".
* However, every factual statement must remain grounded in the Context section below.

FORMATTING
* When a guest asks about a specific room, menu item, or spa treatment for the FIRST time in the conversation, or asks for a general overview of it, structure your answer like this example:

Executive Suite
Price: ₹12000/night
Capacity: 3 guests
Key details:
- Complimentary breakfast
- Executive lounge access
- Airport transfer

Follow this shape (name, then price/duration, then a short bullet list) but only include fields that are actually present in the context — never write "Not specified" or leave a field blank.

* If the guest then asks a narrow follow-up about just one detail of something already introduced (e.g. "does it include airport transfer?", "how much is it?", "how long does that treatment take?"), answer that one detail directly in a plain sentence instead of repeating the full structured block.
* For general or conversational questions (greetings, policy explanations, location questions, directions, contact info), use plain natural sentences instead of the structured template.
* When listing multiple rooms, restaurants, or treatments in response to one question, use one short structured block per item, separated by a blank line.
* Keep prices exactly as provided in the context using ₹ where applicable.
* Keep times, durations, distances, capacities, and other numerical information exactly as stated in the context.
* When asked how to contact the hotel, provide the phone number, email address, or website exactly as given in the context.

Context:
{context}
"""

In [10]:
SOURCE_RELEVANCE_MARGIN = 0.3
CANDIDATE_CEILING = 15
DOC_INCLUSION_MAX_DISTANCE = 1.1
CITATION_MAX_DISTANCE = 0.65
MAX_CONTEXT_CHARS = 12000
MAX_HISTORY_MESSAGES = 10 


def enforce_context_budget(docs, max_chars=MAX_CONTEXT_CHARS):
    trimmed, total = [], 0
    for doc in docs:
        total += len(doc.page_content)
        if total > max_chars and trimmed:
            break
        trimmed.append(doc)
    return trimmed

def retrieve_with_confidence(question):
    collection_size = vectorstore._collection.count()
    k = min(CANDIDATE_CEILING, collection_size)

    result = vectorstore.similarity_search_with_score(question, k=k)
    if not result:
        return [], []

    result = sorted(result, key=lambda x: x[1])
    best_score = result[0][1]

    
    included = [(doc, score) for doc, score in result if score <= DOC_INCLUSION_MAX_DISTANCE]
    if not included:
        included = result[:1]

    
    docs = enforce_context_budget([doc for doc, _ in included])

    
    if best_score > CITATION_MAX_DISTANCE:
        relevant_types = []
    else:
        relevant_types = sorted({
            doc.metadata.get("doc_type", "info")
            for doc, score in included
            if score <= best_score + SOURCE_RELEVANCE_MARGIN
        })

    return docs, relevant_types


def extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict) and "text" in part:
                parts.append(part["text"])
        return "".join(parts)
    return str(content)


def safe_invoke(messages):
    try:
        response = llm.invoke(messages)
        return extract_text(response.content)
    except Exception as e:
        print(f"LLM call failed: {e}")
        return (
            "I'm having trouble responding right now \u2014 please try again in a moment. "
            "If this keeps happening, feel free to reach out to us directly on WhatsApp or Instagram."
        )


In [11]:
ROOMS_BY_TYPE = {r["room_type"]: r for r in catalog_data.get("rooms", [])}
EXTRA_BED_CHARGE = catalog_data.get("policies", {}).get("extra_bed_charge_inr", 0)
print(f"Indexed {len(ROOMS_BY_TYPE)} room type(s) for deterministic cost lookups")

Indexed 4 room type(s) for deterministic cost lookups


In [12]:
NIGHTS_PATTERN = re.compile(r"(\d+)\s*night", re.IGNORECASE)

WORD_TO_NUM = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}
EXTRA_BED_PATTERN = re.compile(
    r"(\d+|one|two|three|four|five)\s*extra\s*bed", re.IGNORECASE
)
BARE_EXTRA_BED_PATTERN = re.compile(r"\bextra\s*bed\b", re.IGNORECASE)


def parse_extra_beds(text):
    match = EXTRA_BED_PATTERN.search(text)
    if match:
        val = match.group(1).lower()
        return int(val) if val.isdigit() else WORD_TO_NUM[val]
    if BARE_EXTRA_BED_PATTERN.search(text):
        return 1
    return 0


def extract_mentioned_room(text):
    for room_name in ROOMS_BY_TYPE:
        if re.search(rf"\b{re.escape(room_name)}\b", text, re.IGNORECASE):
            return room_name
    return None


def compute_stay_cost(text, history_text=""):
    room_name = extract_mentioned_room(text) or extract_mentioned_room(history_text)
    if not room_name:
        return None

    nights_match = NIGHTS_PATTERN.search(text) or NIGHTS_PATTERN.search(history_text)
    if not nights_match:
        return None
    nights = int(nights_match.group(1))

    extra_beds = parse_extra_beds(text) or parse_extra_beds(history_text)

    room = ROOMS_BY_TYPE[room_name]
    room_total = room["price_inr_per_night"] * nights
    extra_bed_total = EXTRA_BED_CHARGE * extra_beds * nights
    grand_total = room_total + extra_bed_total

    lines = [
        f"{room_name}: ₹{room['price_inr_per_night']}/night x {nights} night(s) = ₹{room_total}",
    ]
    if extra_beds:
        lines.append(f"Extra beds: {extra_beds} x ₹{EXTRA_BED_CHARGE}/night x {nights} night(s) = ₹{extra_bed_total}")
    lines.append(f"Total estimated cost: ₹{grand_total}")

    return "Computed check:\n" + "\n".join(lines)

In [13]:
SOURCES_LINE_PATTERN = re.compile(r"\n\n_Sources:.*_$", re.DOTALL)


def strip_sources_line(text):
    return SOURCES_LINE_PATTERN.sub("", text or "")

def normalize_content(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict) and "text" in part:
                parts.append(part["text"])
        return "".join(parts)
    return str(content) if content is not None else ""

def history_to_messages(history):
    messages = []
    if not history:
        return messages
    for turn in history:
        if isinstance(turn, dict):
            role = turn.get("role")
            content = normalize_content(turn.get("content", ""))
            if role == "user":
                messages.append(HumanMessage(content=content))
            elif role == "assistant":
                messages.append(AIMessage(content=strip_sources_line(content)))
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_msg, assistant_msg = turn
            if user_msg:
                messages.append(HumanMessage(content=normalize_content(user_msg)))
            if assistant_msg:
                messages.append(AIMessage(content=strip_sources_line(normalize_content(assistant_msg))))
    return messages[-MAX_HISTORY_MESSAGES:]


def answer_question(question, history):
    docs, relevant_types = retrieve_with_confidence(question)
    past_messages = history_to_messages(history)

    history_text = " ".join(
        normalize_content(turn.get("content", ""))
        for turn in (history or [])
        if isinstance(turn, dict)
    )

    if not docs:
        return safe_invoke([
            SystemMessage(content=(
                "You are the virtual concierge for The Saffron Grand, a luxury 5-star hotel "
                "in Vijay Nagar, Indore. No specific hotel information was found for this message. "
                "If it's small talk or a greeting, respond warmly and invite the guest to ask about "
                "rooms, dining, spa services, facilities, policies, location, parking, or contact details. "
                "If it's a factual question, say you don't have that information and suggest they contact "
                "the hotel directly using the details you have, or ask about something else."
            )),
            *past_messages,
            HumanMessage(content=question),
        ])

    context = "\n\n".join(doc.page_content for doc in docs)
    computed_check = compute_stay_cost(question, history_text)
    if computed_check:
        context = f"{context}\n\n{computed_check}"

    formatted_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    answer_text = safe_invoke([
        SystemMessage(content=formatted_prompt),
        *past_messages,
        HumanMessage(content=question),
    ])

    return answer_text

In [14]:
_real_rooms = dict(ROOMS_BY_TYPE)
ROOMS_BY_TYPE.clear()
ROOMS_BY_TYPE.update({
    "Executive Suite": {"room_type": "Executive Suite", "price_inr_per_night": 12000,
                         "capacity": 3, "amenities": ["Breakfast", "Airport transfer"]},
    "Deluxe Room": {"room_type": "Deluxe Room", "price_inr_per_night": 6000,
                     "capacity": 2, "amenities": ["Breakfast"]},
})

assert compute_stay_cost("Do you have a pool?") is None, \
    "should return None when no room/nights is mentioned"

basic = compute_stay_cost("What's the cost for the Deluxe Room for 3 nights?")
assert basic is not None and "18000" in basic, f"expected ₹18000 total, got: {basic!r}"

with_bed = compute_stay_cost("Executive Suite for 2 nights with 1 extra bed")
assert with_bed is not None and "24000" in with_bed and "Extra beds" in with_bed, \
    f"expected room + extra bed total, got: {with_bed!r}"

ROOMS_BY_TYPE.clear()
ROOMS_BY_TYPE.update(_real_rooms)
print("Layer 1 passed: stay-cost math unit tests.")

Layer 1 passed: stay-cost math unit tests.


In [15]:
demo = gr.ChatInterface(
    fn=answer_question,
    title="The Saffron Grand — Ask us anything",
    description="Rooms, dining, spa, facilities, policies, location, and more.",
)

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [16]:
print(normalize_content)

<function normalize_content at 0x0000019B368F0A40>
